##### Imports

In [1]:
import pandas as pd 
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
import mlflow
import time

# Load cleaned data

In [2]:
df = pd.read_csv('../data/complaints_clean.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nSample:")
print(df.head(3))

Dataset shape: (25000, 4)

Sample:
   complaint_id  ...                                     issue
0       1808066  ...     Cont'd attempts collect debt not owed
1       1709127  ...  Loan servicing, payments, escrow account
2       1440945  ...                          Billing disputes

[3 rows x 4 columns]


# Load embedding model

In [3]:
# Track which model we use — important for MLflow comparison later
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Model loaded: {EMBEDDING_MODEL}")
print(f"Embedding dimensions: {model.get_sentence_embedding_dimension()}")

Loading weights: 100%|████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4847.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: all-MiniLM-L6-v2
Embedding dimensions: 384


/tmp/ipykernel_457615/3430115734.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimensions: {model.get_sentence_embedding_dimension()}")


# Generate embeddings with MLflow tracking

In [4]:
mlflow.set_experiment("semantic-search-cfpb")

2026/04/13 18:12:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/13 18:12:22 INFO mlflow.store.db.utils: Updating database tables
2026/04/13 18:12:25 INFO mlflow.tracking.fluent: Experiment with name 'semantic-search-cfpb' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/spectra/Desktop/ml-ai-journey/Week-8/Project-2-Semantic-Search/notebooks/mlruns/1', creation_time=1776084145093, experiment_id='1', last_update_time=1776084145093, lifecycle_stage='active', name='semantic-search-cfpb', tags={}, trace_location=None, workspace='default'>

In [5]:
with mlflow.start_run(run_name=f"{EMBEDDING_MODEL}-25k"):
    mlflow.log_param("embedding model", EMBEDDING_MODEL)
    mlflow.log_param("dataset_size",len(df))
    mlflow.log_param("embedding_dim", model.get_sentence_embedding_dimension())

    # Encode 
    print(f"Encoding {len(df)} complaints...")
    start_time = time.time()

    embeddings = model.encode(
        df['clean_narrative'].tolist(),
        batch_size=64,
        show_progress_bar=True
    )

    encoding_time = time.time() - start_time
    throughput = len(df) / encoding_time

    mlflow.log_metric("encoding_time_seconds", round(encoding_time,2))
    mlflow.log_metric("throughput_per_second", round(throughput,2))

    print(f"Encoding complete: ")
    print(f"Time: {encoding_time:.1f} seconds")
    print(f"Throughput: {throughput:.0f} sentences/second")
    print(f"Embeddings shape: {embeddings.shape}")

/tmp/ipykernel_457615/3642101375.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  mlflow.log_param("embedding_dim", model.get_sentence_embedding_dimension())


Encoding 25000 complaints...


Batches: 100%|██████████████████████████████████████████████████████████████| 391/391 [11:11<00:00,  1.72s/it]


Encoding complete: 
Time: 671.7 seconds
Throughput: 37 sentences/second
Embeddings shape: (25000, 384)


# Store embeddings in ChromaDB

In [6]:
client = chromadb.PersistentClient(path="../data/chroma_db")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [7]:
try:
    client.delete_collection("cfpb_complaints")
except: 
    pass

collection = client.create_collection(
    name="cfpb_complaints",
    metadata={"hnsw:space" : "cosine"}
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [8]:
batch_size = 1000
print(f"Storing {len(df)} embeddings in ChromaD...")

Storing 25000 embeddings in ChromaD...


In [10]:
for i in range(0,len(df),batch_size):
    batch_end = min(i+batch_size,len(df))

    collection.add(
        documents=df['clean_narrative'].iloc[i:batch_end].tolist(),
        embeddings=embeddings[i:batch_end].tolist(),
        ids= [str(x) for x in df['complaint_id'].iloc[i:batch_end].tolist()],
        metadatas=[
            {
                'product': row['product'],
                'issue': row['issue']
            } for _, row in df.iloc[i:batch_end].iterrows()
        ]
    )

    if (i // batch_size) % 5 == 0:
        print(f"Stored {batch_end}/{len(df)} complaints")

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Stored 1000/25000 complaints
Stored 6000/25000 complaints
Stored 11000/25000 complaints
Stored 16000/25000 complaints


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Stored 21000/25000 complaints


In [11]:
print(f"\nFinal collection size: {collection.count()}")


Final collection size: 25000


# Test semantic search

In [12]:
def search_complaints(query, n_results=5 , filter_product=None):
    # Embed the query 
    query_embedding =  model.encode([query]).tolist()

    # Build filter if product specified 
    where = { "product": filter_product} if filter_product else None

    # Search ChromaDB
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where,
        include=['documents', 'distances', 'metadatas']
    )

    print(f"Query: '{query}'")
    for i , (doc, distance, meta) in enumerate(zip(
        results['documents'][0],
        results['distances'][0],
        results['metadatas'][0]
    )):
        score = 1- distance
        print(f"{i+1}. [{meta['product']}] Score: {score:.4f}")
        print(f"   {doc[:150]}")
        print()

In [13]:
search_complaints("credit card fraud unauthorized charges")
print("="*60)
search_complaints("mortgage payment late fees")
print("="*60)
search_complaints("debt collector harassment phone calls")
Run it and paste output. This is the moment of truth — does semantic search work on real financial complaints?

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: 'credit card fraud unauthorized charges'
1. [Credit card] Score: 0.7457
   Credit card showed unauthorized charges as I RARELY use the card in the amount of AMOUNT. Advise them I did not authorize. Agent confirmed I would not

2. [Credit card] Score: 0.7120
   I opened B of A credit card and within a month I got unauthorized charges. My replacement card that was overnighted to me on REDACTED was showing as r

3. [Credit card] Score: 0.7018
   For weeks, I 've asked this company to replace my card, as it has been compromised. Someone has made unauthorized charges on this card. REDACTED charg

4. [Credit card] Score: 0.6923
   A charge posted on my account in the last billing period that I did n't authorize. The charge came from a merchant whose services are pushed on me thr

5. [Credit card] Score: 0.6708
   My card with Continental Finance was compromised about a year and a half ago, and the issue had been cleared up. About six months later my card was ch

Query: 'mortgage payme